<a href="https://www.kaggle.com/code/muhammaddhiyaulatha/arc-baseline-zero-model-ipynb?scriptVersionId=312558797" target="_blank"><img align="left" alt="Kaggle" title="Open in Kaggle" src="https://kaggle.com/static/images/open-in-kaggle.svg"></a>

# ARC Prize 2026 - Baseline Model

This notebook contains my first submission to the ARC-AGI-2 competition on Kaggle.

## Approach

* Generate output grids filled with zeros
* Match input grid dimensions
* Ensure correct submission format

## Purpose

This is a baseline to understand:

* Submission pipeline
* Evaluation system
* Dataset structure

Next step: implement rule-based reasoning.


In [4]:
import json
import os
from collections import Counter

is_rerun = bool(os.getenv("KAGGLE_IS_COMPETITION_RERUN"))

# ================================
# 📦 LOAD DATA
# ================================
if is_rerun:
    path = "/kaggle/input/competitions/arc-prize-2026-arc-agi-2/arc-agi_test_challenges.json"
else:
    path = "/kaggle/input/competitions/arc-prize-2026-arc-agi-2/arc-agi_evaluation_challenges.json"

with open(path) as f:
    data = json.load(f)

# ================================
# 🧠 HELPER FUNCTIONS
# ================================
def zero_grid(h, w):
    return [[0 for _ in range(w)] for _ in range(h)]

def copy_grid(grid):
    return [row[:] for row in grid]

def most_common_color(grid):
    flat = [c for row in grid for c in row]
    return Counter(flat).most_common(1)[0][0]

def fill_with_color(h, w, color):
    return [[color for _ in range(w)] for _ in range(h)]

def similarity(a, b):
    """Hitung kesamaan grid"""
    if len(a) != len(b) or len(a[0]) != len(b[0]):
        return 0
    total = 0
    same = 0
    for i in range(len(a)):
        for j in range(len(a[0])):
            total += 1
            if a[i][j] == b[i][j]:
                same += 1
    return same / total if total > 0 else 0


# ================================
# 🧠 SOLVER STRATEGY
# ================================
def solve_task(task):
    predictions = []

    # Ambil pola dari training terakhir (kalau ada)
    last_train_output = None
    if len(task["train"]) > 0:
        last_train_output = task["train"][-1]["output"]

    for test_case in task["test"]:
        inp = test_case["input"]
        h = len(inp)
        w = len(inp[0])

        candidates = []

        # 1️⃣ Copy input
        candidates.append(copy_grid(inp))

        # 2️⃣ Zero grid
        candidates.append(zero_grid(h, w))

        # 3️⃣ Fill dengan warna dominan input
        dominant = most_common_color(inp)
        candidates.append(fill_with_color(h, w, dominant))

        # 4️⃣ Gunakan output terakhir training (kalau ukuran sama)
        if last_train_output:
            if len(last_train_output) == h and len(last_train_output[0]) == w:
                candidates.append(copy_grid(last_train_output))

        # ================================
        # 🎯 PILIH 2 TERBAIK
        # ================================
        scored = []

        for c in candidates:
            score = similarity(inp, c)
            scored.append((score, c))

        scored.sort(reverse=True, key=lambda x: x[0])

        best_1 = scored[0][1]
        best_2 = scored[1][1] if len(scored) > 1 else best_1

        predictions.append({
            "attempt_1": best_1,
            "attempt_2": best_2
        })

    return predictions


# ================================
# 📝 BUILD SUBMISSION
# ================================
submission = {}

for task_id, task in data.items():
    submission[task_id] = solve_task(task)

# ================================
# 💾 SAVE
# ================================
with open("submission.json", "w") as f:
    json.dump(submission, f)

print("✅ FINAL submission.json created!")

✅ FINAL submission.json created!
